
# Neural Local Volatility benchmark v3
## Sabate/Gierjatowicz-style neural SDE on March 2024 SPX

This notebook is the third implementation pass of the **neural local-volatility (neural LV)** benchmark. It is designed to answer a more precise question than the earlier pilot:

> Can a Sabate/Gierjatowicz-style neural LV be calibrated convincingly to the same SPX universe used by the classical LV and Heston baselines, and how do its same-day fit, frozen next-day prices, and one-day hedges compare?

The two neural-LV coefficient architectures are available behind one switch:

```python
SIGMA_ARCHITECTURE = "shared"
# SIGMA_ARCHITECTURE = "sabate_piecewise"
```

- **`shared`** uses one time-conditioned network $\sigma_\theta(t,X)$. This is the real-data adaptation used in v1/v2, but v3 fixes the training diagnostics and control-variate architecture.
- **`sabate_piecewise`** implements the paper's Section 2.4 idea: one diffusion network on each interval $[T_{i-1},T_i]$, calibrate maturity $T_i$, then freeze that interval and continue to $T_{i+1}$.

The purpose of putting both behind one switch is to make the architectural choice an **experiment**, not an assumption.

---

## What v3 fixes

The earlier pilot was useful for debugging but was not a serious calibration. v3 addresses the main weaknesses identified after that run:

1. **25 OTM quotes per expiry/settlement slice**, matching the Heston calibration cap rather than 9.
2. The control variate is no longer conditioned on contract features. It has **one output per selected calibration option**, much closer to Sabate's vanilla implementation.
3. The SDE is still trained with the control-variate price estimator, but convergence/checkpointing is based on a **raw Monte Carlo pricing RMSE that does not depend on the hedge network**.
4. Diagnostics explicitly separate:
   - raw Monte Carlo pricing RMSE;
   - control-variate pricing RMSE;
   - control-variate variance reduction.
5. The final same-day diagnostic hierarchy is:
   $$
   \boxed{\text{selected calibration strip}\to\text{full OTM strip}\to\text{complete cleaned chain}.}
   $$
6. Shared-network training uses **maturity-balanced contract batches**, reducing the extra noise created by arbitrary contract mini-batches.
7. Piecewise training follows Sabate's incremental maturity procedure: calibrate the current maturity, freeze the newly fitted diffusion interval, and move forward.
8. CPU thread counts are deliberately conservative and the main pipeline contains **no Matplotlib plotting**, because the previous Windows/VS Code kernel crashes were native-process crashes rather than Python exceptions.
9. Training checkpoints are written during long runs so a crash does not require starting from zero.

---

## What remains intentionally adapted from Sabate

### Forward-normalised state

For each calibration date, define
$$
X_u=\frac{S_u}{F_0(u)}.
$$
With deterministic carry,
$$
dX_u=X_u\sigma(u,X_u)\,dW_u,\qquad X_0=1.
$$
This retains the paper's convenient order-one state scale while using the observed OptionMetrics forward and discount curves.

### Real-data maturity grid

Sabate's synthetic experiments use a small regular maturity grid. Here the actual SPX maturity dates are inserted exactly into a base grid with 96 time steps per year.

### No lookback/exotic branch

The robust-price-bound lookback branch is unrelated to the present vanilla-pricing/one-day-hedging benchmark and remains omitted.

Everything else below is written to make deviations visible rather than silently replacing the paper's methodology.



## 1. Imports, CPU stability and reproducibility

The previous kernel deaths occurred without Python tracebacks. v3 therefore limits OpenMP/MKL/PyTorch threading before PyTorch is imported. This may sacrifice some raw CPU throughput, but it is preferable to losing an overnight calibration.

**ETA** means *estimated time remaining*. The progress bars below show elapsed time and an ETA once enough iterations have completed.


In [1]:

from pathlib import Path
from dataclasses import dataclass
import copy
import gc
import json
import math
import os
import random
import time
import warnings

# Conservative settings for Windows / VS Code / Intel laptop stability.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from tqdm.auto import tqdm

import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch, "xpu"):
        try:
            if torch.xpu.is_available():
                return torch.device("xpu")
        except Exception:
            pass
    return torch.device("cpu")


DEVICE = choose_device()

if DEVICE.type == "cpu":
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

print("PyTorch version:", torch.__version__)
print("Selected device:", DEVICE)
print("CPU logical cores visible:", os.cpu_count())
print("PyTorch intra-op threads:", torch.get_num_threads())


PyTorch version: 2.9.1+cpu
Selected device: cpu
CPU logical cores visible: 8
PyTorch intra-op threads: 1



## 2. Architecture and runtime switches

The default is deliberately **not** Sabate-scale computation on the laptop. First use `diagnostic` to verify the v3 behaviour and estimate real wall-clock speed. Then move to `overnight` or an NVIDIA/HPC environment.

The paper/repository uses 40,000 paths per stochastic-gradient iteration, 20 iterations per epoch and up to 1,000 epochs. That is represented by the `hpc` mode, not by the laptop modes.

For `sabate_piecewise`, `epochs` means epochs **per maturity stage**. Because there can be around forty maturities, even moderate per-stage settings can lead to long total runtimes. The notebook prints a stage-by-stage ETA.


In [2]:

# Main experiment switch requested for v3.
SIGMA_ARCHITECTURE = "shared"          # "shared" or "sabate_piecewise"
RUN_MODE = "diagnostic"                # "smoke", "diagnostic", "overnight", "hpc"

# Set True only if you explicitly want to discard checkpoints.
FORCE_RETRAIN = False

# First concrete transition. Later sections can be wrapped across the month.
EXAMPLE_DATE = "2024-03-01"

MODE = {
    "smoke": {
        "epochs": 4,
        "paths_per_step": 64,
        "steps_per_epoch": 1,
        "quotes_per_maturity_per_step": 2,
        "fixed_raw_eval_paths": 256,
        "cv_eval_paths": 256,
        "cv_eval_contracts": 12,
        "eval_every_sde_epochs": 1,
        "cv_eval_every_raw_checks": 2,
        "patience_raw_checks": None,
        "final_mc_paths": 1_000,
        "final_path_chunk": 500,
        "max_wallclock_hours": None,
        "hedge_refinement_epochs": 0,
    },
    "diagnostic": {
        "epochs": 80,
        "paths_per_step": 512,
        "steps_per_epoch": 4,
        "quotes_per_maturity_per_step": 4,
        "fixed_raw_eval_paths": 2_048,
        "cv_eval_paths": 1_024,
        "cv_eval_contracts": 48,
        "eval_every_sde_epochs": 5,
        "cv_eval_every_raw_checks": 2,
        "patience_raw_checks": 8,
        "final_mc_paths": 20_000,
        "final_path_chunk": 5_000,
        "max_wallclock_hours": 3.0,
        "hedge_refinement_epochs": 10,
    },
    "overnight": {
        "epochs": 300,
        "paths_per_step": 2_048,
        "steps_per_epoch": 8,
        "quotes_per_maturity_per_step": 8,
        "fixed_raw_eval_paths": 8_192,
        "cv_eval_paths": 4_096,
        "cv_eval_contracts": 96,
        "eval_every_sde_epochs": 10,
        "cv_eval_every_raw_checks": 2,
        "patience_raw_checks": 10,
        "final_mc_paths": 100_000,
        "final_path_chunk": 10_000,
        "max_wallclock_hours": 12.0,
        "hedge_refinement_epochs": 20,
    },
    "hpc": {
        "epochs": 1_000,
        "paths_per_step": 40_000,
        "steps_per_epoch": 20,
        "quotes_per_maturity_per_step": None,  # use all selected contracts in shared mode
        "fixed_raw_eval_paths": 40_000,
        "cv_eval_paths": 40_000,
        "cv_eval_contracts": None,
        "eval_every_sde_epochs": 10,
        "cv_eval_every_raw_checks": 2,
        "patience_raw_checks": None,
        "final_mc_paths": 400_000,
        "final_path_chunk": 20_000,
        "max_wallclock_hours": None,
        "hedge_refinement_epochs": 50,
    },
}[RUN_MODE]

# Sabate architecture/training constants retained.
SDE_HIDDEN_LAYERS = 4
SDE_WIDTH = 50
HEDGE_HIDDEN_LAYERS = 3
HEDGE_WIDTH = 30
LEARNING_RATE = 1e-3
SDE_GRAD_CLIP = 5.0
HEDGE_GRAD_CLIP = 3.0
BASE_STEPS_PER_YEAR = 96
DELTA_BUMP_SPX_POINTS = 0.50

# Heston currently caps its OTM calibration set at 25 quotes per slice.
MAX_QUOTES_PER_SLICE = 25

# Diagnostic threshold only; real market mids are not expected to reproduce synthetic errors.
TARGET_RAW_NORMALISED_RMSE = 2e-5

CACHE_DIR = Path("neural_lv_v3_cache")
CACHE_DIR.mkdir(exist_ok=True)

print("SIGMA_ARCHITECTURE =", SIGMA_ARCHITECTURE)
print("RUN_MODE =", RUN_MODE)
print(json.dumps(MODE, indent=2))


SIGMA_ARCHITECTURE = shared
RUN_MODE = diagnostic
{
  "epochs": 80,
  "paths_per_step": 512,
  "steps_per_epoch": 4,
  "quotes_per_maturity_per_step": 4,
  "fixed_raw_eval_paths": 2048,
  "cv_eval_paths": 1024,
  "cv_eval_contracts": 48,
  "eval_every_sde_epochs": 5,
  "cv_eval_every_raw_checks": 2,
  "patience_raw_checks": 8,
  "final_mc_paths": 20000,
  "final_path_chunk": 5000,
  "max_wallclock_hours": 3.0,
  "hedge_refinement_epochs": 10
}



## 3. Load the same four OptionMetrics/IvyDB files


In [4]:

def find_file(pattern):
    search_dirs = [Path("."), Path("/mnt/data")]
    found = []
    for folder in search_dirs:
        found.extend(sorted(folder.glob(pattern)))
    if not found:
        raise FileNotFoundError(f"Could not find a file matching {pattern!r}")
    return found[0]


FILES = {
    "options": find_file("one_month_spx_option_price.csv"),
    "forwards": find_file("one_month_spx_forward_price.csv"),
    "spot": find_file("one_month_spx_security_price.csv"),
    "rates": find_file("one_month_spx_zero_coupon_yield_curve.csv"),
}

for name, path in FILES.items():
    print(f"{name:8s}: {path}")

options_raw = pd.read_csv(FILES["options"], low_memory=False)
forwards_raw = pd.read_csv(FILES["forwards"], low_memory=False)
spot_raw = pd.read_csv(FILES["spot"], low_memory=False)
rates_raw = pd.read_csv(FILES["rates"], low_memory=False)


options : one_month_spx_option_price.csv
forwards: one_month_spx_forward_price.csv
spot    : one_month_spx_security_price.csv
rates   : one_month_spx_zero_coupon_yield_curve.csv



## 4. Exact LV/Heston common cleaning universe

The cleaning rules remain unchanged from the current LV/Heston benchmark:

- SPX `secid = 108105`;
- European, standard 100-multiplier, unadjusted contracts;
- calls and puts;
- 7 to 365 calendar days to expiry;
- positive non-crossed bid/ask;
- market implied-volatility sanity filter;
- $0.05\le |\Delta_{\mathrm{IvyDB}}|\le 0.95$;
- relative spread at most 25%;
- $|\log(K/F)|\le0.30$;
- elementary European option-price bounds.

Calibration uses one OTM quote per strike, with the tighter side preferred close to ATM. A maturity/settlement slice must have at least eight usable OTM observations.


In [5]:

SECID = 108105
MIN_DTE, MAX_DTE = 7, 365
MAX_ABS_LOG_MONEYNESS = 0.30
MIN_ABS_DELTA, MAX_ABS_DELTA = 0.05, 0.95
MAX_RELATIVE_SPREAD = 0.25
ATM_TIGHTER_SIDE_BAND = 0.05
MIN_POINTS_PER_SLICE = 8


def prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw):
    options = options_raw.copy()
    options["date"] = pd.to_datetime(options["date"])
    options["exdate"] = pd.to_datetime(options["exdate"])
    options["K"] = options["strike_price"] / 1000.0
    options["T_days"] = (options["exdate"] - options["date"]).dt.days
    options["T"] = options["T_days"] / 365.0
    options["bid"] = pd.to_numeric(options["best_bid"], errors="coerce")
    options["ask"] = pd.to_numeric(options["best_offer"], errors="coerce")
    options["mid"] = 0.5 * (options["bid"] + options["ask"])
    options["spread"] = options["ask"] - options["bid"]
    options["relative_spread"] = options["spread"] / options["mid"].replace(0, np.nan)
    options["iv_market"] = pd.to_numeric(options["impl_volatility"], errors="coerce")
    options["delta_market"] = pd.to_numeric(options["delta"], errors="coerce")
    options["abs_delta_market"] = options["delta_market"].abs()

    mask = (
        (options["secid"] == SECID)
        & (options["exercise_style"] == "E")
        & (options["contract_size"] == 100)
        & (options["cfadj"] == 1)
        & options["cp_flag"].isin(["C", "P"])
        & (options["bid"] >= 0)
        & (options["ask"] > options["bid"])
        & (options["mid"] > 0)
        & options["T_days"].between(MIN_DTE, MAX_DTE)
        & options["iv_market"].between(1e-4, 5.0)
        & options["abs_delta_market"].between(MIN_ABS_DELTA, MAX_ABS_DELTA)
        & (options["relative_spread"] <= MAX_RELATIVE_SPREAD)
    )
    options = options.loc[mask].copy()

    spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
    spot["date"] = pd.to_datetime(spot["date"])
    spot = spot.rename(columns={"close": "S"}).drop_duplicates("date")

    forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
    forwards["date"] = pd.to_datetime(forwards["date"])
    forwards["expiration"] = pd.to_datetime(forwards["expiration"])
    forwards = forwards.rename(columns={
        "expiration": "exdate",
        "AMSettlement": "am_settlement",
        "ForwardPrice": "F",
    })
    forwards = forwards[
        ["date", "exdate", "am_settlement", "F"]
    ].drop_duplicates(["date", "exdate", "am_settlement"])

    rates = rates_raw.copy()
    rates["date"] = pd.to_datetime(rates["date"])
    rates["days"] = pd.to_numeric(rates["days"], errors="coerce")
    rates["rate_decimal"] = pd.to_numeric(rates["rate"], errors="coerce") / 100.0
    rate_curves = {
        d: (g["days"].to_numpy(float), g["rate_decimal"].to_numpy(float))
        for d, g in rates.dropna().sort_values(["date", "days"]).groupby("date")
    }

    options = options.merge(spot, on="date", how="left").merge(
        forwards,
        on=["date", "exdate", "am_settlement"],
        how="left",
        validate="many_to_one",
    )
    options["r"] = [
        float(np.interp(n, *rate_curves[d])) if d in rate_curves else np.nan
        for d, n in zip(options["date"], options["T_days"])
    ]
    options["D"] = np.exp(-options["r"] * options["T"])
    options["carry"] = np.log(options["F"] / options["S"]) / options["T"]
    options["q"] = options["r"] - options["carry"]
    options["k"] = np.log(options["K"] / options["F"])

    options = options.dropna(subset=["S", "F", "r", "D", "carry", "k"])
    options = options.loc[options["k"].abs() <= MAX_ABS_LOG_MONEYNESS].copy()

    call_lb = options["D"] * np.maximum(options["F"] - options["K"], 0.0)
    call_ub = options["D"] * options["F"]
    put_lb = options["D"] * np.maximum(options["K"] - options["F"], 0.0)
    put_ub = options["D"] * options["K"]
    lb = np.where(options["cp_flag"].eq("C"), call_lb, put_lb)
    ub = np.where(options["cp_flag"].eq("C"), call_ub, put_ub)
    options = options.loc[
        (options["mid"] >= lb - 1e-6) & (options["mid"] <= ub + 1e-6)
    ].copy()

    return options.sort_values(
        ["date", "exdate", "am_settlement", "K", "cp_flag"]
    ).reset_index(drop=True)


def build_otm_surface(quotes):
    idx = [
        "date", "exdate", "am_settlement", "K", "T", "S", "F",
        "r", "q", "carry", "D", "k"
    ]
    vals = [
        "mid", "bid", "ask", "spread", "relative_spread",
        "iv_market", "delta_market", "optionid"
    ]

    wide = quotes.pivot_table(
        index=idx, columns="cp_flag", values=vals, aggfunc="first"
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    strict_call = wide["K"] >= wide["F"]
    near = wide["k"].abs() <= ATM_TIGHTER_SIDE_BAND
    have_c = wide.get("iv_market_C", pd.Series(np.nan, index=wide.index)).notna()
    have_p = wide.get("iv_market_P", pd.Series(np.nan, index=wide.index)).notna()
    c_spread = wide.get("spread_C", pd.Series(np.inf, index=wide.index))
    p_spread = wide.get("spread_P", pd.Series(np.inf, index=wide.index))

    choose_call = strict_call.copy()
    both_near = near & have_c & have_p
    choose_call = np.where(both_near, c_spread <= p_spread, choose_call)
    wide["cp_otm"] = np.where(choose_call, "C", "P")

    for base in vals:
        c = wide.get(f"{base}_C", pd.Series(np.nan, index=wide.index))
        p = wide.get(f"{base}_P", pd.Series(np.nan, index=wide.index))
        wide[f"{base}_otm"] = np.where(choose_call, c, p)

    wide = wide.dropna(subset=["iv_market_otm", "mid_otm", "optionid_otm"]).copy()

    sizes = wide.groupby(["date", "exdate", "am_settlement"]).size()
    valid = sizes[sizes >= MIN_POINTS_PER_SLICE].index

    return (
        wide.set_index(["date", "exdate", "am_settlement"])
        .loc[valid]
        .reset_index()
        .sort_values(["date", "T", "k"])
        .reset_index(drop=True)
    )


quotes_all = prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw)
otm = build_otm_surface(quotes_all)

print("Complete cleaned quotes:", len(quotes_all))
print("OTM calibration observations:", len(otm))
print("Trading dates:", otm["date"].nunique())
print("Expiry/settlement slices:", otm.groupby(["date", "exdate", "am_settlement"]).ngroups)


Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809



## 5. Deterministic forward and discount curves


In [6]:

@dataclass
class DailyMarketCurve:
    date: pd.Timestamp
    am_settlement: int
    spot: float
    T_nodes: np.ndarray
    logD_nodes: np.ndarray
    logF_nodes: np.ndarray

    def __post_init__(self):
        self._logD = PchipInterpolator(self.T_nodes, self.logD_nodes, extrapolate=True)
        self._logF = PchipInterpolator(self.T_nodes, self.logF_nodes, extrapolate=True)

    def log_discount(self, t):
        return self._logD(np.maximum(np.asarray(t, float), 0.0))

    def log_forward_ratio(self, t):
        return self._logF(np.maximum(np.asarray(t, float), 0.0))

    def discount(self, t0, t1):
        return np.exp(self.log_discount(t1) - self.log_discount(t0))

    def forward(self, t):
        return self.spot * np.exp(self.log_forward_ratio(t))

    def q_integral(self, t0, t1):
        return (
            -(self.log_discount(t1) - self.log_discount(t0))
            -(self.log_forward_ratio(t1) - self.log_forward_ratio(t0))
        )


class CurveBook:
    def __init__(self, spot_raw, forwards_raw, rates_raw):
        self.spot = spot_raw.loc[
            spot_raw["secid"] == SECID, ["date", "close"]
        ].copy()
        self.spot["date"] = pd.to_datetime(self.spot["date"])
        self.spot = self.spot.set_index("date")["close"].to_dict()

        self.forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
        self.forwards["date"] = pd.to_datetime(self.forwards["date"])
        self.forwards["expiration"] = pd.to_datetime(self.forwards["expiration"])

        self.rates = rates_raw.copy()
        self.rates["date"] = pd.to_datetime(self.rates["date"])
        self.rates["rate_decimal"] = pd.to_numeric(
            self.rates["rate"], errors="coerce"
        ) / 100.0
        self._cache = {}

    def get(self, date, am_settlement):
        date = pd.Timestamp(date)
        am = int(am_settlement)
        key = (date, am)
        if key in self._cache:
            return self._cache[key]

        S0 = float(self.spot[date])

        rg = self.rates.loc[
            self.rates["date"] == date, ["days", "rate_decimal"]
        ].dropna().sort_values("days")
        T_r = rg["days"].to_numpy(float) / 365.0
        logD = -rg["rate_decimal"].to_numpy(float) * T_r

        fg = self.forwards.loc[
            (self.forwards["date"] == date)
            & (self.forwards["AMSettlement"] == am),
            ["expiration", "ForwardPrice"],
        ].copy()
        if len(fg) < 2:
            fg = self.forwards.loc[
                self.forwards["date"] == date,
                ["expiration", "ForwardPrice"],
            ].copy()

        fg["T"] = (fg["expiration"] - date).dt.days / 365.0
        fg = (
            fg.loc[(fg["T"] >= 0) & (fg["ForwardPrice"] > 0)]
            .groupby("T", as_index=False)["ForwardPrice"]
            .mean()
            .sort_values("T")
        )
        T_f = fg["T"].to_numpy(float)
        logF = np.log(fg["ForwardPrice"].to_numpy(float) / S0)
        positive = T_f > 0
        T_f, logF = T_f[positive], logF[positive]

        T_all = np.unique(np.concatenate([[0.0], T_r, T_f]))
        logD_all = np.interp(T_all, np.r_[0.0, T_r], np.r_[0.0, logD])
        logF_all = np.interp(T_all, np.r_[0.0, T_f], np.r_[0.0, logF])

        curve = DailyMarketCurve(date, am, S0, T_all, logD_all, logF_all)
        self._cache[key] = curve
        return curve


curve_book = CurveBook(spot_raw, forwards_raw, rates_raw)



## 6. Select 25 calibration quotes per expiry/settlement slice

The **cleaned universe is unchanged**. The 25-point cap only defines the subset used inside the expensive neural-SDE optimisation, exactly as the Heston notebook uses a computational calibration subset while retaining the complete cleaned chain for evaluation.

Selection is deterministic and approximately evenly spread in log-moneyness within every expiry/settlement slice.


In [7]:

def sample_calibration_day(day, max_per_slice=MAX_QUOTES_PER_SLICE):
    pieces = []
    for _, g in day.groupby(["exdate", "am_settlement"]):
        g = g.sort_values("k")
        if len(g) <= max_per_slice:
            pieces.append(g)
        else:
            positions = np.unique(
                np.round(np.linspace(0, len(g) - 1, max_per_slice)).astype(int)
            )
            pieces.append(g.iloc[positions])
    return pd.concat(pieces, ignore_index=True)


def prepare_selected_calibration(date):
    date = pd.Timestamp(date)
    day = otm.loc[otm["date"] == date].copy()
    selected = sample_calibration_day(day, MAX_QUOTES_PER_SLICE).copy()
    selected = selected.rename(columns={
        "mid_otm": "target_mid",
        "cp_otm": "target_cp",
        "optionid_otm": "optionid",
    }).reset_index(drop=True)

    # Smoke-mode piecewise runs only a few maturity stages so that the entire
    # notebook can be code-tested quickly. Its final pricing numbers are not
    # research results. Diagnostic/overnight/HPC modes always use all maturities.
    if RUN_MODE == "smoke" and SIGMA_ARCHITECTURE == "sabate_piecewise":
        keep_T = np.sort(selected["T"].unique())[:3]
        selected = selected.loc[selected["T"].isin(keep_T)].reset_index(drop=True)

    selected["contract_id"] = np.arange(len(selected), dtype=int)
    return selected


selected_example = prepare_selected_calibration(EXAMPLE_DATE)
print("Selected calibration contracts:", len(selected_example))
print("Unique maturity times:", selected_example["T"].nunique())
print("Expiry/settlement slices:", selected_example.groupby(["exdate", "am_settlement"]).ngroups)
print("Maximum selected per expiry/settlement slice:", selected_example.groupby(["exdate", "am_settlement"]).size().max())


Selected calibration contracts: 997
Unique maturity times: 36
Expiry/settlement slices: 40
Maximum selected per expiry/settlement slice: 25



## 7. Neural-network building blocks

### 7.1 Diffusion networks

Each diffusion network retains Sabate's architecture:

- input $(t,X_t)$;
- four hidden ReLU layers;
- width 50;
- positive Softplus output.

`SharedNeuralLocalVol` contains one such network.

`SabatePiecewiseLocalVol` contains one such network on every interval $[T_{i-1},T_i]$. This is the paper's equation (2.8) construction.

### 7.2 One-output-per-option control variate

Sabate's vanilla control-variate network assigns a distinct output coordinate to each calibrated option. v3 restores that idea.

The implementation below has a shared hidden trunk and an output weight/bias for every selected option. `forward_ids(...)` computes only the requested output coordinates. This is algebraically the same as evaluating the full output layer and then slicing it, but avoids wasting CPU work on contracts not present in the current balanced batch.


In [8]:

class FeedForwardTrunk(nn.Module):
    def __init__(self, input_dim, hidden_layers, width):
        super().__init__()
        layers = []
        d = input_dim
        for _ in range(hidden_layers):
            layers.append(nn.Linear(d, width))
            layers.append(nn.ReLU())
            d = width
        self.net = nn.Sequential(*layers)
        self.output_dim = width

    def forward(self, x):
        return self.net(x)


def xavier_sabate(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight.data, gain=1.5)
        if module.bias is not None:
            nn.init.zeros_(module.bias.data)


class SigmaNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, SDE_HIDDEN_LAYERS, SDE_WIDTH)
        self.out = nn.Sequential(nn.Linear(SDE_WIDTH, 1), nn.Softplus())

    def forward(self, t, x):
        if x.ndim == 1:
            x = x[:, None]
        if not torch.is_tensor(t):
            t = torch.full_like(x, float(t))
        elif t.ndim == 0:
            t = torch.full_like(x, float(t.item()))
        elif t.ndim == 1:
            t = t[:, None]
        h = self.trunk(torch.cat([t, x], dim=1))
        return self.out(h)


class SharedNeuralLocalVol(nn.Module):
    architecture_name = "shared"

    def __init__(self):
        super().__init__()
        self.sigma_net = SigmaNet()
        self.apply(xavier_sabate)

    def forward(self, t, x):
        return self.sigma_net(t, x)

    def interval_index(self, t):
        return 0


class SabatePiecewiseLocalVol(nn.Module):
    architecture_name = "sabate_piecewise"

    def __init__(self, maturity_boundaries):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.sigma_nets = nn.ModuleList([SigmaNet() for _ in self.boundaries])
        self.apply(xavier_sabate)

    def interval_index(self, t):
        # [0,T1], (T1,T2], ... . For a left-endpoint Euler step at exactly Ti,
        # the next interval should be used, hence side='right'.
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.sigma_nets) - 1)

    def forward(self, t, x):
        return self.sigma_nets[self.interval_index(t)](t, x)

    def freeze_all(self):
        for p in self.parameters():
            p.requires_grad_(False)

    def unfreeze_interval(self, idx):
        self.freeze_all()
        for p in self.sigma_nets[idx].parameters():
            p.requires_grad_(True)


class PerOptionHedgeNet(nn.Module):
    """Sabate-style distinct hedge output per calibrated option."""
    def __init__(self, n_options):
        super().__init__()
        self.n_options = int(n_options)
        self.trunk = FeedForwardTrunk(2, HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH)
        self.output_weight = nn.Parameter(torch.empty(self.n_options, HEDGE_WIDTH))
        self.output_bias = nn.Parameter(torch.zeros(self.n_options))
        self.apply(xavier_sabate)
        nn.init.xavier_normal_(self.output_weight, gain=1.5)

    def forward_ids(self, t, x, option_ids):
        if x.ndim == 1:
            x = x[:, None]
        if not torch.is_tensor(t):
            t = torch.full_like(x, float(t))
        elif t.ndim == 0:
            t = torch.full_like(x, float(t.item()))
        elif t.ndim == 1:
            t = t[:, None]

        h = self.trunk(torch.cat([t, x], dim=1))
        ids = torch.as_tensor(option_ids, dtype=torch.long, device=h.device)
        W = self.output_weight[ids]
        b = self.output_bias[ids]
        return h @ W.T + b[None, :]


class StageHedgeBank(nn.Module):
    """One Algorithm-1 hedge network for each maturity stage in piecewise mode."""
    def __init__(self, stage_sizes):
        super().__init__()
        self.nets = nn.ModuleList([PerOptionHedgeNet(n) for n in stage_sizes])


def set_requires_grad(module, value):
    for p in module.parameters():
        p.requires_grad_(value)



## 8. Time grid and calibration context

Every listed calibration maturity is inserted exactly into the simulation grid. The remaining points start from Sabate's 96-steps-per-year discretisation.


In [9]:

def build_time_grid(maturities, steps_per_year=BASE_STEPS_PER_YEAR):
    maturities = np.sort(np.unique(np.asarray(maturities, float)))
    max_T = float(maturities[-1])
    base = np.arange(0.0, max_T + 0.5 / steps_per_year, 1.0 / steps_per_year)
    base = base[base <= max_T + 1e-12]
    grid = np.unique(np.r_[0.0, base, maturities, max_T])
    grid.sort()
    return grid


def nearest_grid_indices(grid, times):
    grid = np.asarray(grid)
    times = np.asarray(times)
    idx = np.searchsorted(grid, times)
    idx = np.clip(idx, 0, len(grid) - 1)
    left = np.maximum(idx - 1, 0)
    choose_left = np.abs(grid[left] - times) < np.abs(grid[idx] - times)
    idx = np.where(choose_left, left, idx)
    err = np.max(np.abs(grid[idx] - times))
    if err > 1e-10:
        raise RuntimeError(f"Maturity not represented exactly. max error={err}")
    return idx


@dataclass
class CalibrationContext:
    date: pd.Timestamp
    spot: float
    contracts: pd.DataFrame
    maturity_boundaries: np.ndarray
    grid_np: np.ndarray
    grid: torch.Tensor
    maturity_index: torch.Tensor
    stage_index: np.ndarray
    K: torch.Tensor
    T: torch.Tensor
    F_T: torch.Tensor
    D_T: torch.Tensor
    target: torch.Tensor
    cp_sign: torch.Tensor
    am: torch.Tensor
    F_grid: torch.Tensor
    D_grid: torch.Tensor


def build_calibration_context(date):
    contracts = prepare_selected_calibration(date)
    date = pd.Timestamp(date)
    maturity_boundaries = np.sort(contracts["T"].unique())
    grid_np = build_time_grid(maturity_boundaries)
    maturity_index_np = nearest_grid_indices(grid_np, contracts["T"].to_numpy(float))
    stage_map = {round(float(T), 12): i for i, T in enumerate(maturity_boundaries)}
    stage_index = np.array([stage_map[round(float(T), 12)] for T in contracts["T"]], dtype=int)

    S0 = float(contracts["S"].iloc[0])
    F_grid_np = np.empty((2, len(grid_np)))
    D_grid_np = np.empty((2, len(grid_np)))
    for am in [0, 1]:
        curve = curve_book.get(date, am)
        F_grid_np[am] = curve.forward(grid_np)
        D_grid_np[am] = curve.discount(0.0, grid_np)

    cp_sign = np.where(contracts["target_cp"].eq("C"), 1.0, -1.0)

    return CalibrationContext(
        date=date,
        spot=S0,
        contracts=contracts,
        maturity_boundaries=maturity_boundaries,
        grid_np=grid_np,
        grid=torch.tensor(grid_np, dtype=torch.float32, device=DEVICE),
        maturity_index=torch.tensor(maturity_index_np, dtype=torch.long, device=DEVICE),
        stage_index=stage_index,
        K=torch.tensor(contracts["K"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        T=torch.tensor(contracts["T"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        F_T=torch.tensor(contracts["F"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        D_T=torch.tensor(contracts["D"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        target=torch.tensor(contracts["target_mid"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        cp_sign=torch.tensor(cp_sign, dtype=torch.float32, device=DEVICE),
        am=torch.tensor(contracts["am_settlement"].to_numpy(int), dtype=torch.long, device=DEVICE),
        F_grid=torch.tensor(F_grid_np, dtype=torch.float32, device=DEVICE),
        D_grid=torch.tensor(D_grid_np, dtype=torch.float32, device=DEVICE),
    )


calib_ctx = build_calibration_context(EXAMPLE_DATE)
print("Selected contracts:", len(calib_ctx.contracts))
print("Maturity networks if piecewise:", len(calib_ctx.maturity_boundaries))
print("Simulation time steps to longest maturity:", len(calib_ctx.grid_np) - 1)


Selected contracts: 997
Maturity networks if piecewise: 36
Simulation time steps to longest maturity: 129



## 9. Monte Carlo estimators used in training

For option $j$, v3 keeps both estimators:

### Raw estimator
$$
\widehat P_j^{\rm raw}=D(0,T_j)\Phi_j(S_{T_j}).
$$

### Sabate control-variate estimator
$$
\widehat P_j^{\rm cv}
=
D(0,T_j)\Phi_j(S_{T_j})
-
\sum_{k<T_j}
\bar h_j(t_k,X_{t_k})\,
D(0,t_k)S_{t_k}\sigma_{t_k}\Delta W_k.
$$

During an **SDE update**, the second term uses detached state/diffusion values, reproducing the key backward-propagation treatment in Algorithm 1. The SDE loss is the squared error of the **CV price estimator**, because variance reduction is part of Sabate's training method.

During a **hedge update**, the SDE is fixed and the hedge minimises the sample variance of the CV estimator.

However, **checkpoint selection uses raw pricing RMSE**, so a finite-sample non-zero mean of the control variate can no longer make the diffusion network appear better than it really is.


In [10]:

def antithetic_normals(total_paths, n_steps, seed):
    half = (total_paths + 1) // 2
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed))
    z = torch.randn(half, n_steps, generator=gen, dtype=torch.float32)
    z = torch.cat([z, -z], dim=0)[:total_paths]
    return z.to(DEVICE)


def simulate_estimators(model, hedge, ctx, contract_ids, z, phase, hedge_local_ids=None):
    """
    Generic selected-contract estimator for shared training and piecewise stages.

    contract_ids are GLOBAL indices into ctx.contracts.
    hedge_local_ids identify output coordinates in hedge; for the shared hedge these are
    the same as global ids, while a piecewise stage hedge uses 0..stage_size-1.
    """
    ids_np = np.asarray(contract_ids, dtype=int)
    ids = torch.as_tensor(ids_np, dtype=torch.long, device=DEVICE)
    if hedge_local_ids is None:
        hedge_local_ids = ids_np
    hedge_local_ids = np.asarray(hedge_local_ids, dtype=int)

    maturity_idx = ctx.maturity_index[ids]
    K = ctx.K[ids]
    F_T = ctx.F_T[ids]
    D_T = ctx.D_T[ids]
    cp = ctx.cp_sign[ids]
    am = ctx.am[ids]

    B = z.shape[0]
    M = len(ids_np)
    x = torch.ones((B, 1), dtype=torch.float32, device=DEVICE)
    cv = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)
    raw_payoff = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)
    max_step = int(maturity_idx.max().item())

    global_to_local_col = {int(g): j for j, g in enumerate(ids_np)}

    for step in range(max_step):
        t = ctx.grid[step]
        dt = ctx.grid[step + 1] - ctx.grid[step]
        sqrt_dt = torch.sqrt(dt)

        if phase == "hedge":
            with torch.no_grad():
                sigma = model(t, x)
        else:
            sigma = model(t, x)

        dW = sqrt_dt * z[:, step:step + 1]

        active_local_cols = torch.where(maturity_idx > step)[0]
        if len(active_local_cols):
            active_global = ids[active_local_cols]
            active_global_np = active_global.detach().cpu().numpy().astype(int)
            active_hedge_ids = [
                hedge_local_ids[global_to_local_col[int(g)]] for g in active_global_np
            ]

            if phase == "sde":
                with torch.no_grad():
                    h = hedge.forward_ids(t, x.detach(), active_hedge_ids)
            else:
                h = hedge.forward_ids(t, x.detach(), active_hedge_ids)

            stochastic_piece = x.detach() * sigma.detach() * dW
            am_active = am[active_local_cols]
            discounted_forward = (
                ctx.D_grid[am_active, step] * ctx.F_grid[am_active, step]
            )
            contribution = h * stochastic_piece * discounted_forward[None, :]
            cv[:, active_local_cols] = cv[:, active_local_cols] + contribution

        a = x * sigma
        denominator = 1.0 + torch.abs(x.detach() * sigma.detach()) * sqrt_dt
        x = x + a * dW / denominator

        matured_local_cols = torch.where(maturity_idx == step + 1)[0]
        if len(matured_local_cols):
            S_T = x * F_T[matured_local_cols][None, :]
            strike = K[matured_local_cols][None, :]
            call_payoff = torch.clamp(S_T - strike, min=0.0)
            put_payoff = torch.clamp(strike - S_T, min=0.0)
            is_call = (cp[matured_local_cols] > 0)[None, :]
            discounted = torch.where(is_call, call_payoff, put_payoff) * D_T[matured_local_cols][None, :]
            raw_payoff[:, matured_local_cols] = discounted

    cv_estimator = raw_payoff - cv
    return raw_payoff, cv_estimator



## 10. Maturity-balanced batches for the shared architecture

A completely random batch can over-represent some maturities and omit others. v3 samples approximately the same number of contracts from every maturity on every shared-network SGD iteration.

In `hpc` mode, setting `quotes_per_maturity_per_step=None` uses the entire selected calibration strip.


In [11]:

class MaturityBalancedSampler:
    def __init__(self, ctx, per_maturity, seed=SEED):
        self.ctx = ctx
        self.per_maturity = per_maturity
        self.rng = np.random.default_rng(seed)
        self.groups = [
            np.where(ctx.stage_index == j)[0]
            for j in range(len(ctx.maturity_boundaries))
        ]

    def draw(self):
        if self.per_maturity is None:
            return np.arange(len(self.ctx.contracts), dtype=int)
        chosen = []
        for ids in self.groups:
            n = min(int(self.per_maturity), len(ids))
            chosen.extend(self.rng.choice(ids, size=n, replace=False).tolist())
        return np.asarray(chosen, dtype=int)



## 11. Raw-Monte-Carlo evaluation independent of the hedge network

This is the key v3 diagnostic fix.

The function below simulates the diffusion **without evaluating the control variate at all** and prices every selected calibration option from the resulting terminal samples. The same fixed Brownian shocks are reused at every diagnostic check.

Therefore, if raw RMSE improves, the change must come from the diffusion model rather than from a finite-sample mean of the learned hedge.


In [12]:

@torch.no_grad()
def simulate_terminal_x_selected(model, ctx, z, max_stage=None):
    if max_stage is None:
        max_stage = len(ctx.maturity_boundaries) - 1
    max_T = ctx.maturity_boundaries[max_stage]
    max_step = int(nearest_grid_indices(ctx.grid_np, [max_T])[0])
    x = torch.ones((z.shape[0], 1), dtype=torch.float32, device=DEVICE)

    terminal = {}
    stage_by_grid_idx = {
        int(nearest_grid_indices(ctx.grid_np, [T])[0]): j
        for j, T in enumerate(ctx.maturity_boundaries[: max_stage + 1])
    }

    for step in range(max_step):
        t = ctx.grid[step]
        dt = ctx.grid[step + 1] - ctx.grid[step]
        sqrt_dt = torch.sqrt(dt)
        sigma = model(t, x)
        dW = sqrt_dt * z[:, step:step + 1]
        a = x * sigma
        denominator = 1.0 + torch.abs(a) * sqrt_dt
        x = x + a * dW / denominator

        grid_idx = step + 1
        if grid_idx in stage_by_grid_idx:
            terminal[stage_by_grid_idx[grid_idx]] = x.squeeze(1).detach().cpu().numpy().copy()

    return terminal


def empirical_payoff_mean(samples, strike_norm, cp_flag):
    x = np.sort(np.asarray(samples, float))
    n = len(x)
    prefix = np.r_[0.0, np.cumsum(x)]
    total = prefix[-1]
    k = np.asarray(strike_norm, float)
    idx = np.searchsorted(x, k, side="right")
    below_sum = prefix[idx]
    above_sum = total - below_sum
    call = (above_sum - k * (n - idx)) / n
    put = (k * idx - below_sum) / n
    return np.where(np.asarray(cp_flag) == "C", call, put)


def prices_for_selected_from_terminal(ctx, terminal, upto_stage=None):
    if upto_stage is None:
        upto_stage = len(ctx.maturity_boundaries) - 1
    rows = []
    for stage in range(upto_stage + 1):
        ids = np.where(ctx.stage_index == stage)[0]
        g = ctx.contracts.iloc[ids]
        payoff = empirical_payoff_mean(
            terminal[stage],
            g["K"].to_numpy(float) / g["F"].to_numpy(float),
            g["target_cp"].to_numpy(),
        )
        price = g["D"].to_numpy(float) * g["F"].to_numpy(float) * payoff
        tmp = g[["contract_id", "T", "k", "target_cp", "target_mid"]].copy()
        tmp["raw_model_price"] = price
        tmp["raw_error"] = price - tmp["target_mid"].to_numpy(float)
        rows.append(tmp)
    return pd.concat(rows, ignore_index=True)


def raw_selected_diagnostics(model, ctx, z_fixed, upto_stage=None):
    terminal = simulate_terminal_x_selected(model, ctx, z_fixed, max_stage=upto_stage)
    table = prices_for_selected_from_terminal(ctx, terminal, upto_stage=upto_stage)
    err = table["raw_error"].to_numpy(float)
    return {
        "raw_rmse": float(np.sqrt(np.mean(err ** 2))),
        "raw_mae": float(np.mean(np.abs(err))),
        "raw_normalised_rmse": float(np.sqrt(np.mean(err ** 2)) / ctx.spot),
        "table": table,
    }



## 12. Control-variate diagnostics kept separate from raw model fit

For a selected contract subset, v3 reports:

- **CV RMSE**: the finite-Monte-Carlo price estimator after subtracting the learned trading control variate;
- **median variance ratio**
  $$
  \mathrm{median}_j\frac{\operatorname{Var}(\widehat P_j^{cv})}
  {\operatorname{Var}(\widehat P_j^{raw})}.
  $$

A ratio below 1 means variance reduction. These numbers measure control-variate quality, but they do **not** select the diffusion checkpoint.


In [13]:

def choose_even_ids(ids, n):
    ids = np.asarray(ids, dtype=int)
    if n is None or n >= len(ids):
        return ids
    pos = np.unique(np.round(np.linspace(0, len(ids) - 1, n)).astype(int))
    return ids[pos]


@torch.no_grad()
def cv_diagnostics_shared(model, hedge, ctx, ids, z):
    raw, cv = simulate_estimators(model, hedge, ctx, ids, z, phase="sde")
    ids_t = torch.as_tensor(ids, dtype=torch.long, device=DEVICE)
    target = ctx.target[ids_t]
    raw_price = raw.mean(dim=0)
    cv_price = cv.mean(dim=0)
    raw_var = raw.var(dim=0, unbiased=False)
    cv_var = cv.var(dim=0, unbiased=False)
    return {
        "raw_rmse_subset": float(torch.sqrt(torch.mean((raw_price - target) ** 2)).cpu()),
        "cv_rmse_subset": float(torch.sqrt(torch.mean((cv_price - target) ** 2)).cpu()),
        "median_variance_ratio": float(torch.median(cv_var / torch.clamp(raw_var, min=1e-12)).cpu()),
    }


@torch.no_grad()
def cv_diagnostics_piecewise(model, hedge_bank, ctx, z_full, max_contracts=None):
    ratios = []
    raw_errors = []
    cv_errors = []
    for stage in range(len(ctx.maturity_boundaries)):
        global_ids = np.where(ctx.stage_index == stage)[0]
        chosen_global = choose_even_ids(global_ids, max_contracts)
        local_lookup = {g: i for i, g in enumerate(global_ids)}
        chosen_local = np.array([local_lookup[g] for g in chosen_global], dtype=int)
        max_step = int(nearest_grid_indices(ctx.grid_np, [ctx.maturity_boundaries[stage]])[0])
        z = z_full[:, :max_step]
        raw, cv = simulate_estimators(
            model,
            hedge_bank.nets[stage],
            ctx,
            chosen_global,
            z,
            phase="sde",
            hedge_local_ids=chosen_local,
        )
        ids_t = torch.as_tensor(chosen_global, dtype=torch.long, device=DEVICE)
        target = ctx.target[ids_t]
        raw_price = raw.mean(dim=0)
        cv_price = cv.mean(dim=0)
        raw_var = raw.var(dim=0, unbiased=False)
        cv_var = cv.var(dim=0, unbiased=False)
        ratios.extend((cv_var / torch.clamp(raw_var, min=1e-12)).cpu().numpy().tolist())
        raw_errors.extend((raw_price - target).cpu().numpy().tolist())
        cv_errors.extend((cv_price - target).cpu().numpy().tolist())

    raw_errors = np.asarray(raw_errors)
    cv_errors = np.asarray(cv_errors)
    return {
        "raw_rmse_subset": float(np.sqrt(np.mean(raw_errors ** 2))),
        "cv_rmse_subset": float(np.sqrt(np.mean(cv_errors ** 2))),
        "median_variance_ratio": float(np.median(ratios)),
    }



## 13. Checkpoint helpers and runtime profiler

Long neural-SDE calibrations should not depend on one uninterrupted notebook kernel.

- Shared mode saves a resumable checkpoint after every raw diagnostic check.
- Piecewise mode saves after every completed maturity stage and also writes a temporary current-stage checkpoint.
- The first few optimisation iterations record forward/backward wall time and print a crude projected runtime. This is a measurement from **your machine**, not a guessed runtime.


In [14]:

def checkpoint_file(date, architecture):
    return CACHE_DIR / f"{pd.Timestamp(date):%Y%m%d}_{architecture}_{RUN_MODE}.pt"


def hours_since(t0):
    return (time.time() - t0) / 3600.0


def wallclock_exceeded(t0):
    limit = MODE["max_wallclock_hours"]
    return limit is not None and hours_since(t0) >= float(limit)


def print_runtime_projection(samples, remaining_updates, label="training"):
    if not samples:
        return
    sec = float(np.median(samples[-min(5, len(samples)):]))
    hours = sec * max(0, remaining_updates) / 3600.0
    print(f"Measured median optimisation-step time: {sec:.2f}s")
    print(f"Crude {label} time remaining at this speed: {hours:.2f} hours")



## 14. Shared-network training

This mode retains one global $\sigma_\theta(t,X)$.

The two alternating epoch types are exactly the Sabate idea:

- **SDE epoch**: freeze the hedge, update $\theta$ using the CV pricing loss;
- **hedge epoch**: freeze the SDE, minimise CV estimator variance.

The important v3 difference is diagnostic: every `eval_every_sde_epochs` SDE epochs, raw fixed-path RMSE is computed without the hedge and used for checkpoint selection.


In [15]:

def train_shared(ctx):
    path = checkpoint_file(ctx.date, "shared")
    model = SharedNeuralLocalVol().to(DEVICE)
    hedge = PerOptionHedgeNet(len(ctx.contracts)).to(DEVICE)
    opt_sde = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    opt_hedge = torch.optim.Adam(hedge.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(opt_sde, milestones=[500, 800], gamma=0.2)

    start_epoch = 0
    history = []
    best_raw = np.inf
    best_model_state = copy.deepcopy(model.state_dict())
    best_hedge_state = copy.deepcopy(hedge.state_dict())
    no_improve = 0

    if path.exists() and not FORCE_RETRAIN:
        payload = torch.load(path, map_location=DEVICE)
        if payload.get("completed", False):
            model.load_state_dict(payload["best_model_state"])
            hedge.load_state_dict(payload["best_hedge_state"])
            print("Loaded completed shared checkpoint:", path)
            return model, hedge, pd.DataFrame(payload["history"])
        model.load_state_dict(payload["model_state"])
        hedge.load_state_dict(payload["hedge_state"])
        opt_sde.load_state_dict(payload["opt_sde"])
        opt_hedge.load_state_dict(payload["opt_hedge"])
        scheduler.load_state_dict(payload["scheduler"])
        start_epoch = int(payload["next_epoch"])
        history = payload["history"]
        best_raw = float(payload["best_raw"])
        best_model_state = payload["best_model_state"]
        best_hedge_state = payload["best_hedge_state"]
        no_improve = int(payload["no_improve"])
        print(f"Resuming shared checkpoint at epoch {start_epoch + 1}:", path)

    sampler = MaturityBalancedSampler(
        ctx, MODE["quotes_per_maturity_per_step"], seed=SEED + 10
    )
    z_raw = antithetic_normals(
        MODE["fixed_raw_eval_paths"], len(ctx.grid_np) - 1, SEED + 100
    )
    z_cv = antithetic_normals(
        MODE["cv_eval_paths"], len(ctx.grid_np) - 1, SEED + 200
    )
    all_ids = np.arange(len(ctx.contracts), dtype=int)
    cv_ids = choose_even_ids(all_ids, MODE["cv_eval_contracts"])

    sde_epoch_count = sum(1 for e in range(start_epoch) if e % 2 == 0)
    raw_check_count = 0
    step_times = []
    t0 = time.time()
    stop_reason = None

    bar = tqdm(range(start_epoch, MODE["epochs"]), desc="Shared neural LV", unit="epoch")

    for epoch in bar:
        phase = "sde" if epoch % 2 == 0 else "hedge"
        set_requires_grad(model, phase == "sde")
        set_requires_grad(hedge, phase == "hedge")
        losses = []

        for _ in range(MODE["steps_per_epoch"]):
            ids = sampler.draw()
            z = torch.randn(
                MODE["paths_per_step"], len(ctx.grid_np) - 1,
                dtype=torch.float32, device=DEVICE
            )
            opt_sde.zero_grad(set_to_none=True)
            opt_hedge.zero_grad(set_to_none=True)
            tic = time.time()
            raw, cv = simulate_estimators(model, hedge, ctx, ids, z, phase=phase)
            ids_t = torch.as_tensor(ids, dtype=torch.long, device=DEVICE)

            if phase == "sde":
                price_cv = cv.mean(dim=0)
                target = ctx.target[ids_t]
                loss = torch.mean(((price_cv - target) / ctx.spot) ** 2)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), SDE_GRAD_CLIP)
                opt_sde.step()
            else:
                loss = torch.var(cv / ctx.spot, dim=0, unbiased=False).mean()
                loss.backward()
                nn.utils.clip_grad_norm_(hedge.parameters(), HEDGE_GRAD_CLIP)
                opt_hedge.step()

            step_times.append(time.time() - tic)
            losses.append(float(loss.detach().cpu()))
            del raw, cv, z

        scheduler.step()

        raw_diag = None
        cv_diag = None
        if phase == "sde":
            sde_epoch_count += 1
            if sde_epoch_count % MODE["eval_every_sde_epochs"] == 0 or epoch == start_epoch:
                set_requires_grad(model, False)
                set_requires_grad(hedge, False)
                raw_diag = raw_selected_diagnostics(model, ctx, z_raw)
                raw_check_count += 1

                if raw_diag["raw_rmse"] < best_raw:
                    best_raw = raw_diag["raw_rmse"]
                    best_model_state = copy.deepcopy(model.state_dict())
                    best_hedge_state = copy.deepcopy(hedge.state_dict())
                    no_improve = 0
                else:
                    no_improve += 1

                if raw_check_count % MODE["cv_eval_every_raw_checks"] == 0:
                    cv_diag = cv_diagnostics_shared(model, hedge, ctx, cv_ids, z_cv)

                payload = {
                    "completed": False,
                    "next_epoch": epoch + 1,
                    "model_state": model.state_dict(),
                    "hedge_state": hedge.state_dict(),
                    "opt_sde": opt_sde.state_dict(),
                    "opt_hedge": opt_hedge.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "history": history,
                    "best_raw": best_raw,
                    "best_model_state": best_model_state,
                    "best_hedge_state": best_hedge_state,
                    "no_improve": no_improve,
                }
                torch.save(payload, path)

                if raw_diag["raw_normalised_rmse"] < TARGET_RAW_NORMALISED_RMSE:
                    stop_reason = "raw RMSE target reached"
                patience = MODE["patience_raw_checks"]
                if patience is not None and no_improve >= patience:
                    stop_reason = f"raw RMSE did not improve for {patience} checks"

        row = {
            "epoch": epoch + 1,
            "phase": phase,
            "train_loss": float(np.mean(losses)),
            "raw_rmse": np.nan if raw_diag is None else raw_diag["raw_rmse"],
            "raw_mae": np.nan if raw_diag is None else raw_diag["raw_mae"],
            "cv_rmse_subset": np.nan if cv_diag is None else cv_diag["cv_rmse_subset"],
            "median_variance_ratio": np.nan if cv_diag is None else cv_diag["median_variance_ratio"],
            "elapsed_hours": hours_since(t0),
        }
        history.append(row)

        postfix = {"phase": phase, "loss": f"{row['train_loss']:.2e}"}
        if raw_diag is not None:
            postfix["raw_RMSE"] = f"{raw_diag['raw_rmse']:.3f}"
        if cv_diag is not None:
            postfix["CV/raw var"] = f"{cv_diag['median_variance_ratio']:.2f}"
        bar.set_postfix(postfix)

        if len(step_times) == 5:
            remaining = (MODE["epochs"] - epoch - 1) * MODE["steps_per_epoch"]
            print_runtime_projection(step_times, remaining, label="shared-mode")

        if wallclock_exceeded(t0):
            stop_reason = f"wall-clock budget {MODE['max_wallclock_hours']}h reached"

        if stop_reason:
            print("Stopping shared training:", stop_reason)
            break

        gc.collect()

    model.load_state_dict(best_model_state)
    hedge.load_state_dict(best_hedge_state)

    # Optional final hedge-only refinement with the best raw-RMSE diffusion frozen.
    # This cannot change pricing fit; it only improves the Sabate control variate.
    refine_epochs = int(MODE.get("hedge_refinement_epochs", 0))
    if refine_epochs > 0 and not wallclock_exceeded(t0):
        set_requires_grad(model, False)
        set_requires_grad(hedge, True)
        refine_opt = torch.optim.Adam(hedge.parameters(), lr=LEARNING_RATE)
        refine_bar = tqdm(range(refine_epochs), desc="Final CV refinement", unit="epoch", leave=False)
        for _ in refine_bar:
            refine_losses = []
            for _ in range(MODE["steps_per_epoch"]):
                ids = sampler.draw()
                z = torch.randn(MODE["paths_per_step"], len(ctx.grid_np)-1, dtype=torch.float32, device=DEVICE)
                refine_opt.zero_grad(set_to_none=True)
                _, cv_ref = simulate_estimators(model, hedge, ctx, ids, z, phase="hedge")
                loss_ref = torch.var(cv_ref/ctx.spot, dim=0, unbiased=False).mean()
                loss_ref.backward()
                nn.utils.clip_grad_norm_(hedge.parameters(), HEDGE_GRAD_CLIP)
                refine_opt.step()
                refine_losses.append(float(loss_ref.detach().cpu()))
                del cv_ref, z
            refine_bar.set_postfix({"variance_loss": f"{np.mean(refine_losses):.2e}"})
        best_hedge_state = copy.deepcopy(hedge.state_dict())

    model.eval(); hedge.eval()
    set_requires_grad(model, False); set_requires_grad(hedge, False)

    torch.save({
        "completed": stop_reason is None or "target" in str(stop_reason) or "did not improve" in str(stop_reason),
        "best_model_state": best_model_state,
        "best_hedge_state": best_hedge_state,
        "history": history,
        "best_raw": best_raw,
        "stop_reason": stop_reason,
    }, path)

    return model, hedge, pd.DataFrame(history)



## 15. Sabate piecewise incremental training

This mode implements the paper's Section 2.4 procedure rather than merely using multiple networks simultaneously.

For each ordered maturity $T_i$:

1. simulate through the already-fitted earlier intervals and the current interval;
2. calibrate the **current maturity's option prices** using Algorithm-1-style alternating SDE/CV updates;
3. select the best current interval network using **raw current-maturity RMSE**;
4. freeze $\sigma_i$;
5. move to $T_{i+1}$.

Earlier diffusion intervals cannot be changed by later stages. This removes the shared model's cross-maturity interference, but it also gives the model substantially more parameters and can fit noisy market cross-sections more aggressively.

Each maturity stage has its own one-output-per-option hedge network, consistent with Algorithm 1's maturity-by-maturity calibration. Those stage hedges are retained so the Sabate-native hedge can later be inspected on selected calibration contracts.


In [16]:

def train_piecewise(ctx):
    path = checkpoint_file(ctx.date, "sabate_piecewise")
    model = SabatePiecewiseLocalVol(ctx.maturity_boundaries).to(DEVICE)
    stage_global_ids = [np.where(ctx.stage_index == j)[0] for j in range(len(ctx.maturity_boundaries))]
    hedge_bank = StageHedgeBank([len(ids) for ids in stage_global_ids]).to(DEVICE)

    completed_stage = -1
    history = []

    if path.exists() and not FORCE_RETRAIN:
        payload = torch.load(path, map_location=DEVICE)
        model.load_state_dict(payload["model_state"])
        hedge_bank.load_state_dict(payload["hedge_bank_state"])
        completed_stage = int(payload.get("completed_stage", -1))
        history = payload.get("history", [])
        if payload.get("completed", False):
            print("Loaded completed piecewise checkpoint:", path)
            model.freeze_all(); model.eval(); hedge_bank.eval()
            return model, hedge_bank, pd.DataFrame(history)
        print(f"Resuming after completed stage {completed_stage + 1} of {len(stage_global_ids)}")

    z_raw_full = antithetic_normals(
        MODE["fixed_raw_eval_paths"], len(ctx.grid_np) - 1, SEED + 300
    )
    z_cv_full = antithetic_normals(
        MODE["cv_eval_paths"], len(ctx.grid_np) - 1, SEED + 400
    )

    t_all = time.time()
    step_times = []
    stop_all = False

    stage_bar = tqdm(
        range(completed_stage + 1, len(stage_global_ids)),
        desc="Sabate maturity stages", unit="stage"
    )

    for stage in stage_bar:
        global_ids = stage_global_ids[stage]
        local_ids = np.arange(len(global_ids), dtype=int)
        hedge = hedge_bank.nets[stage]
        model.unfreeze_interval(stage)
        set_requires_grad(hedge, True)

        opt_sde = torch.optim.Adam(model.sigma_nets[stage].parameters(), lr=LEARNING_RATE)
        opt_hedge = torch.optim.Adam(hedge.parameters(), lr=LEARNING_RATE)
        scheduler = torch.optim.lr_scheduler.MultiStepLR(opt_sde, milestones=[500, 800], gamma=0.2)

        best_stage_raw = np.inf
        best_sigma_state = copy.deepcopy(model.sigma_nets[stage].state_dict())
        best_hedge_state = copy.deepcopy(hedge.state_dict())
        no_improve = 0
        sde_epochs = 0
        raw_checks = 0
        max_step = int(nearest_grid_indices(ctx.grid_np, [ctx.maturity_boundaries[stage]])[0])
        z_raw = z_raw_full[:, :max_step]
        z_cv = z_cv_full[:, :max_step]

        inner = tqdm(range(MODE["epochs"]), desc=f"T={ctx.maturity_boundaries[stage]:.4f}", unit="epoch", leave=False)
        stop_stage = None

        for epoch in inner:
            phase = "sde" if epoch % 2 == 0 else "hedge"
            model.freeze_all()
            set_requires_grad(hedge, phase == "hedge")
            if phase == "sde":
                for p in model.sigma_nets[stage].parameters():
                    p.requires_grad_(True)

            losses = []
            for _ in range(MODE["steps_per_epoch"]):
                z = torch.randn(
                    MODE["paths_per_step"], max_step,
                    dtype=torch.float32, device=DEVICE
                )
                opt_sde.zero_grad(set_to_none=True)
                opt_hedge.zero_grad(set_to_none=True)
                tic = time.time()
                raw, cv = simulate_estimators(
                    model, hedge, ctx, global_ids, z, phase=phase,
                    hedge_local_ids=local_ids
                )
                target = ctx.target[torch.as_tensor(global_ids, dtype=torch.long, device=DEVICE)]

                if phase == "sde":
                    cv_price = cv.mean(dim=0)
                    loss = torch.mean(((cv_price - target) / ctx.spot) ** 2)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.sigma_nets[stage].parameters(), SDE_GRAD_CLIP)
                    opt_sde.step()
                else:
                    loss = torch.var(cv / ctx.spot, dim=0, unbiased=False).mean()
                    loss.backward()
                    nn.utils.clip_grad_norm_(hedge.parameters(), HEDGE_GRAD_CLIP)
                    opt_hedge.step()

                step_times.append(time.time() - tic)
                losses.append(float(loss.detach().cpu()))
                del raw, cv, z

            scheduler.step()

            raw_diag = None
            cv_diag = None
            if phase == "sde":
                sde_epochs += 1
                if sde_epochs % MODE["eval_every_sde_epochs"] == 0 or epoch == 0:
                    model.freeze_all(); set_requires_grad(hedge, False)
                    # Current-maturity raw diagnostic only, exactly what determines this stage.
                    terminal = simulate_terminal_x_selected(model, ctx, z_raw_full, max_stage=stage)
                    table_current = prices_for_selected_from_terminal(ctx, terminal, upto_stage=stage)
                    table_current = table_current.loc[
                        np.isclose(table_current["T"], ctx.maturity_boundaries[stage])
                    ].copy()
                    err = table_current["raw_error"].to_numpy(float)
                    raw_diag = {
                        "raw_rmse": float(np.sqrt(np.mean(err ** 2))),
                        "raw_mae": float(np.mean(np.abs(err))),
                    }
                    raw_checks += 1

                    if raw_diag["raw_rmse"] < best_stage_raw:
                        best_stage_raw = raw_diag["raw_rmse"]
                        best_sigma_state = copy.deepcopy(model.sigma_nets[stage].state_dict())
                        best_hedge_state = copy.deepcopy(hedge.state_dict())
                        no_improve = 0
                    else:
                        no_improve += 1

                    if raw_checks % MODE["cv_eval_every_raw_checks"] == 0:
                        raw_cv, cv_cv = simulate_estimators(
                            model, hedge, ctx, global_ids, z_cv, phase="sde",
                            hedge_local_ids=local_ids
                        )
                        target = ctx.target[torch.as_tensor(global_ids, dtype=torch.long, device=DEVICE)]
                        raw_price = raw_cv.mean(dim=0)
                        cv_price = cv_cv.mean(dim=0)
                        raw_var = raw_cv.var(dim=0, unbiased=False)
                        cv_var = cv_cv.var(dim=0, unbiased=False)
                        cv_diag = {
                            "cv_rmse": float(torch.sqrt(torch.mean((cv_price-target)**2)).cpu()),
                            "median_variance_ratio": float(torch.median(cv_var/torch.clamp(raw_var,min=1e-12)).cpu()),
                        }
                        del raw_cv, cv_cv

                    # Temporary current-stage checkpoint.
                    torch.save({
                        "completed": False,
                        "completed_stage": stage - 1,
                        "model_state": model.state_dict(),
                        "hedge_bank_state": hedge_bank.state_dict(),
                        "history": history,
                        "current_stage": stage,
                        "current_stage_best_raw": best_stage_raw,
                    }, path)

                    patience = MODE["patience_raw_checks"]
                    if patience is not None and no_improve >= patience:
                        stop_stage = f"no raw improvement for {patience} checks"

            history.append({
                "stage": stage + 1,
                "maturity": float(ctx.maturity_boundaries[stage]),
                "epoch_in_stage": epoch + 1,
                "phase": phase,
                "train_loss": float(np.mean(losses)),
                "current_maturity_raw_rmse": np.nan if raw_diag is None else raw_diag["raw_rmse"],
                "cv_rmse": np.nan if cv_diag is None else cv_diag["cv_rmse"],
                "median_variance_ratio": np.nan if cv_diag is None else cv_diag["median_variance_ratio"],
                "elapsed_hours_total": hours_since(t_all),
            })

            postfix = {"phase": phase, "loss": f"{np.mean(losses):.2e}"}
            if raw_diag is not None:
                postfix["raw_RMSE"] = f"{raw_diag['raw_rmse']:.3f}"
            inner.set_postfix(postfix)

            if len(step_times) == 5:
                remaining_stages = len(stage_global_ids) - stage - 1
                remaining_updates_this = (MODE["epochs"] - epoch - 1) * MODE["steps_per_epoch"]
                rough_future = remaining_stages * MODE["epochs"] * MODE["steps_per_epoch"]
                print_runtime_projection(step_times, remaining_updates_this + rough_future, label="piecewise-mode")

            if wallclock_exceeded(t_all):
                stop_stage = f"wall-clock budget {MODE['max_wallclock_hours']}h reached"
                stop_all = True

            if stop_stage:
                break

            gc.collect()

        # Restore best current interval and corresponding maturity hedge.
        model.sigma_nets[stage].load_state_dict(best_sigma_state)
        hedge.load_state_dict(best_hedge_state)
        for p in model.sigma_nets[stage].parameters():
            p.requires_grad_(False)

        # With the best current diffusion interval frozen, spend a small number of
        # extra epochs refining only this maturity's Algorithm-1 control variate.
        refine_epochs = int(MODE.get("hedge_refinement_epochs", 0))
        if refine_epochs > 0 and not wallclock_exceeded(t_all):
            model.freeze_all(); set_requires_grad(hedge, True)
            refine_opt = torch.optim.Adam(hedge.parameters(), lr=LEARNING_RATE)
            for _ in tqdm(range(refine_epochs), desc="CV refinement", unit="epoch", leave=False):
                for _ in range(MODE["steps_per_epoch"]):
                    z = torch.randn(MODE["paths_per_step"], max_step, dtype=torch.float32, device=DEVICE)
                    refine_opt.zero_grad(set_to_none=True)
                    _, cv_ref = simulate_estimators(
                        model, hedge, ctx, global_ids, z, phase="hedge", hedge_local_ids=local_ids
                    )
                    loss_ref = torch.var(cv_ref/ctx.spot, dim=0, unbiased=False).mean()
                    loss_ref.backward()
                    nn.utils.clip_grad_norm_(hedge.parameters(), HEDGE_GRAD_CLIP)
                    refine_opt.step()
                    del cv_ref, z
            best_hedge_state = copy.deepcopy(hedge.state_dict())

        set_requires_grad(hedge, False)

        # Cumulative raw RMSE confirms that earlier maturities have not been damaged.
        cumulative = raw_selected_diagnostics(model, ctx, z_raw_full, upto_stage=stage)
        print(
            f"Finished maturity {stage+1}/{len(stage_global_ids)} "
            f"T={ctx.maturity_boundaries[stage]:.4f}: "
            f"current best raw RMSE=${best_stage_raw:.4f}, "
            f"cumulative selected raw RMSE=${cumulative['raw_rmse']:.4f}"
        )

        torch.save({
            "completed": False,
            "completed_stage": stage,
            "model_state": model.state_dict(),
            "hedge_bank_state": hedge_bank.state_dict(),
            "history": history,
        }, path)

        if stop_all:
            print("Stopping piecewise training due to wall-clock limit. Resume from checkpoint later.")
            break

    completed_stage = max([h["stage"] for h in history], default=0) - 1
    fully_completed = completed_stage >= len(stage_global_ids) - 1 and not stop_all
    model.freeze_all(); model.eval(); hedge_bank.eval()

    torch.save({
        "completed": fully_completed,
        "completed_stage": completed_stage,
        "model_state": model.state_dict(),
        "hedge_bank_state": hedge_bank.state_dict(),
        "history": history,
    }, path)

    return model, hedge_bank, pd.DataFrame(history)



## 16. Train or resume the selected architecture

The first date is 1 March 2024. `diagnostic` is intentionally a **measurement/calibration-development run**, not the final Sabate-scale experiment. The runtime profiler should guide whether the serious run belongs on the laptop overnight or on Imperial HPC/NVIDIA hardware.


In [17]:

calibration_date = pd.Timestamp(EXAMPLE_DATE)
available_dates = sorted(pd.to_datetime(quotes_all["date"].unique()))
if calibration_date not in available_dates:
    raise ValueError(f"{calibration_date.date()} not found")
pos = available_dates.index(calibration_date)
next_date = available_dates[pos + 1]
print("Calibration date:", calibration_date.date())
print("Next trading date:", next_date.date())
print("Calendar-day gap:", (next_date - calibration_date).days)

calib_ctx = build_calibration_context(calibration_date)

if SIGMA_ARCHITECTURE == "shared":
    model, hedge_object, training_history = train_shared(calib_ctx)
elif SIGMA_ARCHITECTURE == "sabate_piecewise":
    model, hedge_object, training_history = train_piecewise(calib_ctx)
else:
    raise ValueError("SIGMA_ARCHITECTURE must be 'shared' or 'sabate_piecewise'")

print("Training/history rows:", len(training_history))
print(training_history.tail(12).to_string(index=False))


Calibration date: 2024-03-01
Next trading date: 2024-03-04
Calendar-day gap: 3


Shared neural LV:   0%|          | 0/80 [00:00<?, ?epoch/s]

Final CV refinement:   0%|          | 0/10 [00:00<?, ?epoch/s]

Training/history rows: 80
 epoch phase  train_loss  raw_rmse  raw_mae  cv_rmse_subset  median_variance_ratio  elapsed_hours
    69   sde    0.000006 11.228100 8.232655        9.905926                0.58829       0.025403
    70 hedge    0.000300       NaN      NaN             NaN                    NaN       0.025777
    71   sde    0.000005       NaN      NaN             NaN                    NaN       0.026120
    72 hedge    0.000310       NaN      NaN             NaN                    NaN       0.026453
    73   sde    0.000005       NaN      NaN             NaN                    NaN       0.026804
    74 hedge    0.000305       NaN      NaN             NaN                    NaN       0.027193
    75   sde    0.000005       NaN      NaN             NaN                    NaN       0.027527
    76 hedge    0.000287       NaN      NaN             NaN                    NaN       0.027861
    77   sde    0.000006       NaN      NaN             NaN                    NaN       0.0


## 17. Final training-quality diagnostic before headline pricing

This section answers three separate questions:

1. **Did the diffusion fit the selected contracts?** Raw fixed-path RMSE on the 25-per-slice calibration set.
2. **Did the control variate learn anything useful?** CV RMSE and median variance ratio on selected contracts.
3. **Where are errors concentrated?** Raw selected error by maturity bucket and moneyness bucket.

If the selected raw RMSE is still poor, there is no reason to interpret next-day results economically yet.


In [18]:

z_final_diag = antithetic_normals(
    MODE["fixed_raw_eval_paths"], len(calib_ctx.grid_np) - 1, SEED + 500
)
selected_raw_diag = raw_selected_diagnostics(model, calib_ctx, z_final_diag)
print("Selected calibration strip raw RMSE: $", round(selected_raw_diag["raw_rmse"], 4))
print("Selected calibration strip raw MAE:  $", round(selected_raw_diag["raw_mae"], 4))

z_cv_diag = antithetic_normals(
    MODE["cv_eval_paths"], len(calib_ctx.grid_np) - 1, SEED + 600
)
if SIGMA_ARCHITECTURE == "shared":
    ids = choose_even_ids(np.arange(len(calib_ctx.contracts)), MODE["cv_eval_contracts"])
    cv_diag = cv_diagnostics_shared(model, hedge_object, calib_ctx, ids, z_cv_diag)
else:
    per_stage_cap = None
    if MODE["cv_eval_contracts"] is not None:
        per_stage_cap = max(1, int(math.ceil(MODE["cv_eval_contracts"] / len(calib_ctx.maturity_boundaries))))
    cv_diag = cv_diagnostics_piecewise(
        model, hedge_object, calib_ctx, z_cv_diag, max_contracts=per_stage_cap
    )

print("CV diagnostic subset raw RMSE: $", round(cv_diag["raw_rmse_subset"], 4))
print("CV diagnostic subset CV RMSE:  $", round(cv_diag["cv_rmse_subset"], 4))
print("Median CV/raw variance ratio:", round(cv_diag["median_variance_ratio"], 4))

selected_error_table = selected_raw_diag["table"].copy()
selected_error_table["maturity_bucket"] = pd.cut(
    selected_error_table["T"], bins=[0, 30/365, 90/365, 180/365, 366/365],
    labels=["7-30d", "31-90d", "91-180d", "181-365d"], include_lowest=True
)
selected_error_table["moneyness_bucket"] = pd.cut(
    selected_error_table["k"], bins=[-0.31, -0.10, -0.03, 0.03, 0.10, 0.31],
    labels=["deep put wing", "put wing", "near ATM", "call wing", "deep call wing"]
)

by_maturity = selected_error_table.groupby("maturity_bucket", observed=True)["raw_error"].agg(
    n="size", RMSE=lambda x: float(np.sqrt(np.mean(np.asarray(x)**2))), MAE=lambda x: float(np.mean(np.abs(x)))
)
by_money = selected_error_table.groupby("moneyness_bucket", observed=True)["raw_error"].agg(
    n="size", RMSE=lambda x: float(np.sqrt(np.mean(np.asarray(x)**2))), MAE=lambda x: float(np.mean(np.abs(x)))
)
print("\nSelected calibration error by maturity bucket:")
print(by_maturity.to_string())
print("\nSelected calibration error by moneyness bucket:")
print(by_money.to_string())


Selected calibration strip raw RMSE: $ 11.3964
Selected calibration strip raw MAE:  $ 8.2975
CV diagnostic subset raw RMSE: $ 10.4872
CV diagnostic subset CV RMSE:  $ 10.5159
Median CV/raw variance ratio: 0.5226

Selected calibration error by maturity bucket:
                   n       RMSE        MAE
maturity_bucket                           
7-30d            225   3.267217   2.832432
31-90d           347   5.368136   4.378790
91-180d          200  11.853463  10.527737
181-365d         225  19.886967  17.823612

Selected calibration error by moneyness bucket:
                    n       RMSE        MAE
moneyness_bucket                           
deep put wing     196  17.289786  16.614808
put wing          330   7.463724   6.491649
near ATM          363   8.687025   4.809647
call wing          89  15.230271  10.142142
deep call wing     19  15.346315  11.857825



## 18. Final raw Monte Carlo simulation for same-day prices and model delta

The full-chain final evaluation does **not** need the control variate. Once $\sigma$ is calibrated, all strikes at a maturity can be priced efficiently from the same empirical terminal distribution.

For delta, the calibrated local-volatility function is frozen in its original day-$t$ coordinate system. We bump the initial spot by $\pm0.5$ SPX points, which corresponds to initial normalised states
$$
X_0^{\pm}=\frac{S_0\pm0.5}{S_0},
$$
and reuse the same Brownian shocks. This is a common-random-number bump-and-revalue delta.


In [19]:

def tkey(t):
    return round(float(t), 12)


@torch.no_grad()
def simulate_same_day_x_samples(model, maturities, total_paths, spot, bump, chunk_paths, seed):
    grid = build_time_grid(maturities)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)
    collected = {key: [] for key in maturity_idx}

    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)
    done = 0
    bar = tqdm(total=total_paths, desc="Final same-day MC", unit="path")

    while done < total_paths:
        n = min(chunk_paths, total_paths - done)
        half = (n + 1) // 2
        z_half = torch.randn(half, len(grid)-1, generator=gen, dtype=torch.float32)
        z = torch.cat([z_half, -z_half], dim=0)[:n].to(DEVICE)
        x0 = torch.tensor(
            [1.0, (spot+bump)/spot, (spot-bump)/spot],
            dtype=torch.float32, device=DEVICE
        )
        x = x0[None, :].repeat(n, 1)

        for step in range(len(grid)-1):
            t = float(grid[step])
            dt = float(grid[step+1]-grid[step])
            sqrt_dt = math.sqrt(dt)
            xflat = x.reshape(n*3, 1)
            tflat = torch.full_like(xflat, t)
            sigma = model(tflat, xflat).reshape(n,3)
            dW = sqrt_dt * z[:, step:step+1]
            a = x * sigma
            x = x + a*dW/(1.0 + torch.abs(a)*sqrt_dt)
            idx = step+1
            if idx in index_to_keys:
                arr = x.detach().cpu().numpy()
                for key in index_to_keys[idx]:
                    collected[key].append(arr.copy())
        done += n
        bar.update(n)
        del z, x
        gc.collect()

    bar.close()
    return {k: np.concatenate(v, axis=0) for k,v in collected.items()}


def price_rows_from_same_day_samples(rows, samples_by_T, base_col="mid"):
    out = rows.copy().reset_index(drop=True)
    base_price = np.empty(len(out))
    up_price = np.empty(len(out))
    down_price = np.empty(len(out))

    for T, idx in out.groupby("T").groups.items():
        pos = np.asarray(list(idx), dtype=int)
        g = out.loc[idx]
        sample = samples_by_T[tkey(T)]
        K_over_F = g["K"].to_numpy(float) / g["F"].to_numpy(float)
        cp = g["cp_flag"].to_numpy()
        scale = g["D"].to_numpy(float) * g["F"].to_numpy(float)
        base_price[pos] = scale * empirical_payoff_mean(sample[:,0], K_over_F, cp)
        up_price[pos] = scale * empirical_payoff_mean(sample[:,1], K_over_F, cp)
        down_price[pos] = scale * empirical_payoff_mean(sample[:,2], K_over_F, cp)

    out["neural_lv_price"] = base_price
    out["neural_lv_delta"] = (up_price-down_price)/(2*DELTA_BUMP_SPX_POINTS)
    out["pricing_error"] = out["neural_lv_price"] - out[base_col]
    out["inside_spread"] = out["neural_lv_price"].between(out["bid"], out["ask"])
    return out



## 19. Same-day diagnostic hierarchy: selected → full OTM → full chain

This is the main localisation tool missing from v1/v2.

- If **selected** RMSE is poor: optimisation/model training is the problem.
- If selected is good but **full OTM** is poor: 25-point subsampling/generalisation across strikes is the problem.
- If full OTM is good but **full chain** deteriorates sharply: investigate call/put parity, ITM evaluation and market-data conventions.


In [20]:

day0_full = quotes_all.loc[quotes_all["date"] == calibration_date].copy().reset_index(drop=True)
day0_otm = otm.loc[otm["date"] == calibration_date].copy().reset_index(drop=True)

samples_same_day = simulate_same_day_x_samples(
    model,
    maturities=day0_full["T"].unique(),
    total_paths=MODE["final_mc_paths"],
    spot=float(day0_full["S"].iloc[0]),
    bump=DELTA_BUMP_SPX_POINTS,
    chunk_paths=MODE["final_path_chunk"],
    seed=SEED+700,
)

# Complete call/put chain.
same_day_full = price_rows_from_same_day_samples(day0_full, samples_same_day, base_col="mid")

# Full OTM strip needs column names compatible with the generic pricer.
otm_eval = day0_otm.rename(columns={
    "mid_otm":"mid", "bid_otm":"bid", "ask_otm":"ask", "cp_otm":"cp_flag", "optionid_otm":"optionid"
}).copy()
same_day_otm = price_rows_from_same_day_samples(otm_eval, samples_same_day, base_col="mid")

# Selected strip: merge exact selected option IDs into full-chain model prices.
selected_ids = set(calib_ctx.contracts["optionid"].astype(int).tolist())
same_day_selected = same_day_full.loc[same_day_full["optionid"].astype(int).isin(selected_ids)].copy()


def price_summary(df):
    e = df["pricing_error"].to_numpy(float)
    return {
        "n": len(df),
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "MAE": float(np.mean(np.abs(e))),
        "inside_bid_ask": float(df["inside_spread"].mean()),
    }

hierarchy = pd.DataFrame({
    "selected_25_per_slice": price_summary(same_day_selected),
    "full_OTM_strip": price_summary(same_day_otm),
    "complete_clean_chain": price_summary(same_day_full),
}).T

print("Same-day pricing hierarchy:")
print(hierarchy.to_string())


Final same-day MC:   0%|          | 0/20000 [00:00<?, ?path/s]

Same-day pricing hierarchy:
                            n       RMSE        MAE  inside_bid_ask
selected_25_per_slice   997.0  10.936081   8.241946        0.028084
full_OTM_strip         4304.0  10.680584   8.364792        0.016264
complete_clean_chain   9046.0  14.584098  11.517727        0.029847



## 20. Same-day error localisation on the complete cleaned chain


In [21]:

loc = same_day_full.copy()
loc["maturity_bucket"] = pd.cut(
    loc["T"], bins=[0,30/365,90/365,180/365,366/365],
    labels=["7-30d","31-90d","91-180d","181-365d"], include_lowest=True
)
loc["moneyness_bucket"] = pd.cut(
    loc["k"], bins=[-0.31,-0.10,-0.03,0.03,0.10,0.31],
    labels=["deep put wing","put wing","near ATM","call wing","deep call wing"]
)


def grouped_error(df, group):
    return df.groupby(group, observed=True)["pricing_error"].agg(
        n="size",
        RMSE=lambda x: float(np.sqrt(np.mean(np.asarray(x)**2))),
        MAE=lambda x: float(np.mean(np.abs(x))),
        bias="mean",
    )

print("By call/put:")
print(grouped_error(loc, "cp_flag").to_string())
print("\nBy maturity bucket:")
print(grouped_error(loc, "maturity_bucket").to_string())
print("\nBy moneyness bucket:")
print(grouped_error(loc, "moneyness_bucket").to_string())


By call/put:
            n       RMSE        MAE      bias
cp_flag                                      
C        4848  15.836814  12.489935 -9.021155
P        4198  12.987940  10.394987 -0.636696

By maturity bucket:
                    n       RMSE        MAE      bias
maturity_bucket                                      
7-30d            1647   4.722311   4.119302 -1.477549
31-90d           2936   9.271022   7.963868 -3.758970
91-180d          3056  16.336146  14.398225 -7.855252
181-365d         1407  24.132578  21.337582 -6.348186

By moneyness bucket:
                     n       RMSE        MAE       bias
moneyness_bucket                                       
deep put wing     2282  21.945793  20.540246 -20.540246
put wing          2952  10.713646   9.191848  -8.700598
near ATM          2959   8.473545   6.077031   4.714210
call wing          793  17.379917  13.848202  13.848202
deep call wing      60  23.862274  20.310399  20.310399



## 21. Identical-contract next-day pairs

The benchmark keeps only options observed on both consecutive trading dates with the same option ID, call/put flag, strike, expiration and settlement convention.


In [22]:

def build_evaluation_pairs(quotes):
    trading_dates = sorted(quotes["date"].unique())
    next_map = {trading_dates[i]: trading_dates[i+1] for i in range(len(trading_dates)-1)}
    left = quotes.copy()
    left["next_date"] = left["date"].map(next_map)
    cols = [
        "date","optionid","cp_flag","K","exdate","am_settlement",
        "mid","bid","ask","S","F","D","r","q","carry","T","k"
    ]
    right = quotes[cols].rename(columns={
        "date":"next_date","mid":"mid_next","bid":"bid_next","ask":"ask_next",
        "S":"S_next","F":"F_next","D":"D_next","r":"r_next","q":"q_next",
        "carry":"carry_next","T":"T_next","k":"k_next",
    })
    return left.merge(
        right,
        on=["next_date","optionid","cp_flag","K","exdate","am_settlement"],
        how="inner", validate="many_to_one"
    )


evaluation_pairs = build_evaluation_pairs(quotes_all)
transition_pairs = evaluation_pairs.loc[
    (evaluation_pairs["date"] == calibration_date)
    & (evaluation_pairs["next_date"] == next_date)
].copy().reset_index(drop=True)
print("Identical-contract pairs:", len(transition_pairs))


Identical-contract pairs: 8771



## 22. Frozen day-$t$ neural-LV next-day pricing

The model is **not recalibrated** on day $t+1$.

Let $h$ be the calendar time elapsed between valuation dates and let
$$
Y_v=\frac{S_{t+1+v}}{F_{t+1}(v)}.
$$
The frozen day-$t$ model was calibrated in the old coordinate
$$
X_{\rm old}(h+v)=\frac{S_{t+1+v}}{F_t(h+v)}
=\frac{F_{t+1}(v)}{F_t(h+v)}Y_v.
$$
Hence the day-$t+1$ simulation evaluates the frozen network at that old coordinate while using tomorrow's spot/forward/discount information for pricing.


In [23]:

@torch.no_grad()
def simulate_next_day_y_samples(
    model, curve0, curve1, elapsed, maturities_remaining,
    total_paths, chunk_paths, seed
):
    grid = build_time_grid(maturities_remaining)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities_remaining, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)
    collected = {key: [] for key in maturity_idx}
    gen = torch.Generator(device="cpu"); gen.manual_seed(seed)
    done = 0
    bar = tqdm(total=total_paths, desc=f"Next-day MC AM={curve1.am_settlement}", unit="path")

    while done < total_paths:
        n = min(chunk_paths, total_paths-done)
        half=(n+1)//2
        zh=torch.randn(half,len(grid)-1,generator=gen,dtype=torch.float32)
        z=torch.cat([zh,-zh],dim=0)[:n].to(DEVICE)
        y=torch.ones((n,1),dtype=torch.float32,device=DEVICE)

        for step in range(len(grid)-1):
            v=float(grid[step]); dt=float(grid[step+1]-grid[step]); sqrt_dt=math.sqrt(dt)
            abs_t=elapsed+v
            Fnew=float(curve1.forward(v)); Fold=float(curve0.forward(abs_t))
            x_old=y*(Fnew/Fold)
            sigma=model(torch.full_like(y,abs_t),x_old)
            dW=sqrt_dt*z[:,step:step+1]
            a=y*sigma
            y=y+a*dW/(1.0+torch.abs(a)*sqrt_dt)
            idx=step+1
            if idx in index_to_keys:
                arr=y.squeeze(1).detach().cpu().numpy()
                for key in index_to_keys[idx]:
                    collected[key].append(arr.copy())
        done+=n; bar.update(n)
        del z,y; gc.collect()
    bar.close()
    return {k:np.concatenate(v) for k,v in collected.items()}


def price_next_day_from_samples(pairs, samples_by_am):
    out=pairs.copy().reset_index(drop=True)
    price=np.empty(len(out))
    for (am,T),idx in out.groupby(["am_settlement","T_next"]).groups.items():
        pos=np.asarray(list(idx),dtype=int); g=out.loc[idx]
        samples=samples_by_am[int(am)][tkey(T)]
        strike_norm=g["K"].to_numpy(float)/g["F_next"].to_numpy(float)
        payoff=empirical_payoff_mean(samples,strike_norm,g["cp_flag"].to_numpy())
        price[pos]=g["D_next"].to_numpy(float)*g["F_next"].to_numpy(float)*payoff
    out["model_price_next"]=price
    out["next_day_pricing_error"]=price-out["mid_next"].to_numpy(float)
    out["next_day_inside_bid_ask"]=out["model_price_next"].between(out["bid_next"],out["ask_next"])
    return out


elapsed=(next_date-calibration_date).days/365.0
samples_next_by_am={}
for am in sorted(transition_pairs["am_settlement"].unique()):
    g=transition_pairs.loc[transition_pairs["am_settlement"]==am]
    curve0=curve_book.get(calibration_date,int(am)); curve1=curve_book.get(next_date,int(am))
    samples_next_by_am[int(am)]=simulate_next_day_y_samples(
        model,curve0,curve1,elapsed,g["T_next"].unique(),
        MODE["final_mc_paths"],MODE["final_path_chunk"],SEED+800+int(am)
    )

next_day_results=price_next_day_from_samples(transition_pairs,samples_next_by_am)
e=next_day_results["next_day_pricing_error"].to_numpy(float)
print("Next-day n:",len(next_day_results))
print("Next-day RMSE: $",round(float(np.sqrt(np.mean(e**2))),4))
print("Next-day MAE:  $",round(float(np.mean(np.abs(e))),4))
print("Next-day inside bid-ask:",round(float(next_day_results["next_day_inside_bid_ask"].mean()),4))


Next-day MC AM=0:   0%|          | 0/20000 [00:00<?, ?path/s]

Next-day MC AM=1:   0%|          | 0/20000 [00:00<?, ?path/s]

Next-day n: 8771
Next-day RMSE: $ 14.7623
Next-day MAE:  $ 11.5708
Next-day inside bid-ask: 0.0204



## 23. Headline model-delta hedge on the complete pair universe

The model delta is the apples-to-apples hedge against BS/LV/Heston. The same one-day self-financing accounting convention is reused.


In [24]:

delta_lookup=same_day_full[["optionid","neural_lv_delta"]].drop_duplicates("optionid")
hedge_results=next_day_results.merge(delta_lookup,on="optionid",how="left",validate="many_to_one")


def one_day_hedging_error(rows, hedge_ratio):
    rows=rows.copy().reset_index(drop=True)
    hedge_ratio=np.asarray(hedge_ratio,float)
    q_integral=np.empty(len(rows)); cash_interest=np.empty(len(rows))
    dt=(rows["next_date"].iloc[0]-rows["date"].iloc[0]).days/365.0
    for am,idx in rows.groupby("am_settlement").groups.items():
        pos=np.asarray(list(idx),dtype=int)
        curve0=curve_book.get(rows.loc[idx,"date"].iloc[0],int(am))
        q_integral[pos]=float(curve0.q_integral(0.0,dt))
        cash0=rows.loc[idx,"mid"].to_numpy(float)-hedge_ratio[pos]*rows.loc[idx,"S"].to_numpy(float)
        cash_interest[pos]=cash0*(1.0/float(curve0.discount(0.0,dt))-1.0)
    dividend_income=hedge_ratio*rows["S"].to_numpy(float)*q_integral
    return (
        rows["mid_next"].to_numpy(float)-rows["mid"].to_numpy(float)
        -hedge_ratio*(rows["S_next"].to_numpy(float)-rows["S"].to_numpy(float))
        -dividend_income-cash_interest
    )


hedge_results["delta_hedging_error"]=one_day_hedging_error(
    hedge_results,hedge_results["neural_lv_delta"]
)
hedge_results["unhedged_change"]=hedge_results["mid_next"]-hedge_results["mid"]

def hedge_summary(error,unhedged):
    error=np.asarray(error,float); unhedged=np.asarray(unhedged,float)
    mae=float(np.mean(np.abs(error))); umae=float(np.mean(np.abs(unhedged)))
    return {
        "n":len(error),"RMSE":float(np.sqrt(np.mean(error**2))),"MAE":mae,
        "bias":float(np.mean(error)),"unhedged_MAE":umae,"improvement":1-mae/umae
    }

print("Model-delta one-day hedge:")
print(pd.Series(hedge_summary(hedge_results["delta_hedging_error"],hedge_results["unhedged_change"])).to_string())


Model-delta one-day hedge:
n               8771.000000
RMSE               0.946163
MAE                0.727375
bias              -0.434842
unhedged_MAE       4.059167
improvement        0.820807



## 24. Additional Sabate-native learned hedge on selected calibration contracts

Because v3 restores **one hedge output per selected calibration option**, the learned control-variate hedge is naturally defined on those selected contracts, not on every unselected quote in the full chain.

Therefore this section compares, on the same selected surviving contracts:

- learned Sabate control-variate hedge;
- model delta.

This is a secondary architecture/training diagnostic. The headline benchmark remains model delta on the complete common pair universe.


In [25]:

@torch.no_grad()
def learned_cv_hedges_selected(model, hedge_object, ctx):
    values=np.empty(len(ctx.contracts))
    x0=torch.ones((1,1),dtype=torch.float32,device=DEVICE)
    t0=torch.zeros_like(x0)
    if SIGMA_ARCHITECTURE=="shared":
        for start in range(0,len(ctx.contracts),256):
            ids=np.arange(start,min(start+256,len(ctx.contracts)))
            values[ids]=hedge_object.forward_ids(t0,x0,ids).squeeze(0).cpu().numpy()
    else:
        for stage in range(len(ctx.maturity_boundaries)):
            gids=np.where(ctx.stage_index==stage)[0]
            lids=np.arange(len(gids))
            values[gids]=hedge_object.nets[stage].forward_ids(t0,x0,lids).squeeze(0).cpu().numpy()
    return values


selected_hedges=calib_ctx.contracts[["optionid","contract_id"]].copy()
selected_hedges["learned_cv_hedge"]=learned_cv_hedges_selected(model,hedge_object,calib_ctx)

selected_pairs=hedge_results.merge(
    selected_hedges[["optionid","learned_cv_hedge"]],on="optionid",how="inner",validate="many_to_one"
).reset_index(drop=True)
selected_pairs["cv_hedging_error"]=one_day_hedging_error(selected_pairs,selected_pairs["learned_cv_hedge"])
selected_pairs["delta_hedging_error_selected"]=one_day_hedging_error(selected_pairs,selected_pairs["neural_lv_delta"])

comparison=pd.DataFrame({
    "model_delta_selected":hedge_summary(selected_pairs["delta_hedging_error_selected"],selected_pairs["unhedged_change"]),
    "learned_CV_selected":hedge_summary(selected_pairs["cv_hedging_error"],selected_pairs["unhedged_change"]),
}).T
print("Selected-contract hedge comparison:")
print(comparison.to_string())

if len(selected_pairs)>2:
    corr=float(np.corrcoef(
        selected_pairs["neural_lv_delta"].to_numpy(float),
        selected_pairs["learned_cv_hedge"].to_numpy(float)
    )[0,1])
    print("Correlation(delta, learned CV hedge):",round(corr,4))


Selected-contract hedge comparison:
                          n      RMSE       MAE      bias  unhedged_MAE  improvement
model_delta_selected  950.0  0.774608  0.624349 -0.428324      1.613947     0.613154
learned_CV_selected   950.0  0.824356  0.666796 -0.354136      1.613947     0.586854


: 

: 


## 25. Compact final summary and exports

No large DataFrame rendering or plotting is required. The important outputs are printed and small CSVs are saved for later analysis.


In [ ]:

final_summary={
    "architecture":SIGMA_ARCHITECTURE,
    "run_mode":RUN_MODE,
    "selected_same_day_RMSE":float(hierarchy.loc["selected_25_per_slice","RMSE"]),
    "full_OTM_same_day_RMSE":float(hierarchy.loc["full_OTM_strip","RMSE"]),
    "complete_chain_same_day_RMSE":float(hierarchy.loc["complete_clean_chain","RMSE"]),
    "next_day_RMSE":float(np.sqrt(np.mean(next_day_results["next_day_pricing_error"].to_numpy(float)**2))),
    "next_day_MAE":float(np.mean(np.abs(next_day_results["next_day_pricing_error"].to_numpy(float)))),
    "delta_hedge_RMSE":float(np.sqrt(np.mean(hedge_results["delta_hedging_error"].to_numpy(float)**2))),
    "delta_hedge_MAE":float(np.mean(np.abs(hedge_results["delta_hedging_error"].to_numpy(float)))),
    "cv_median_variance_ratio":float(cv_diag["median_variance_ratio"]),
}

print(json.dumps(final_summary,indent=2))

stem=f"{calibration_date:%Y%m%d}_{SIGMA_ARCHITECTURE}_{RUN_MODE}"
pd.DataFrame([final_summary]).to_csv(CACHE_DIR/f"summary_{stem}.csv",index=False)
hierarchy.to_csv(CACHE_DIR/f"same_day_hierarchy_{stem}.csv")
training_history.to_csv(CACHE_DIR/f"training_history_{stem}.csv",index=False)
selected_error_table.to_csv(CACHE_DIR/f"selected_error_{stem}.csv",index=False)

print("Saved summary files under",CACHE_DIR.resolve())



## 26. How to compare the two architectures fairly

Run the notebook twice with the same `RUN_MODE` and seed:

```python
SIGMA_ARCHITECTURE = "shared"
```

then

```python
SIGMA_ARCHITECTURE = "sabate_piecewise"
```

Do **not** compare one architecture in `diagnostic` mode with the other in `overnight` or `hpc` mode and interpret the difference as architecture alone.

The comparison table should ultimately contain:

| Dimension | Shared | Sabate piecewise |
|---|---:|---:|
| wall-clock calibration time | | |
| selected same-day raw RMSE | | |
| full OTM same-day RMSE | | |
| complete-chain same-day RMSE | | |
| median CV/raw variance ratio | | |
| next-day RMSE | | |
| model-delta hedge RMSE | | |
| learned-CV hedge RMSE on selected pairs | | |

### Interpretation

If piecewise wins strongly on selected same-day fit but not next-day pricing, that would be evidence for the compute/flexibility-versus-generalisation trade-off discussed before v3.

If shared cannot fit the selected strip even after substantial computation while piecewise can, cross-maturity interference becomes a serious suspect.

If neither can fit the selected strip, the next debugging target is **training/estimator implementation or computational budget**, not the economic dynamics of local volatility.



## 27. Source fidelity notes

The design above follows the uploaded Gierjatowicz/Sabate paper and repository in the aspects that matter for this benchmark:

- Algorithm 1 alternates neural-SDE calibration and control-variate/hedge variance minimisation.
- The SDE update uses a detached copy of the path inside the hedge/control-variate term.
- The numerical scheme is tamed Euler.
- Section 2.4 splits the SDE coefficient into one network per maturity interval and proposes incremental calibration: fit maturity $T_i$, freeze $\sigma_i$, move to $T_{i+1}$.
- The repository's vanilla control variate has separate output coordinates for the calibration options.
- The repository uses Adam at $10^{-3}$, gradient clipping 5 for the SDE and 3 for the control variate, 40,000 paths per SGD batch, 20 batches per epoch and up to 1,000 epochs.

The real-data adaptations are explicitly marked earlier: forward normalisation, a shared-network alternative, irregular real maturities, Heston-matched 25-point slice sampling, and laptop/HPC runtime modes.
